## Part 1

**URL collector (plain `requests`).** Walks the Malay Mail archive pages `malaymail.com/news/malaysia/{year}/{month}` for 2015-2025, reads the paginator to find how many `?pgno=` pages a month has, and parses every article card (`div.col-md-3.article-item`) for `keyword | date | title | url | article_img`. Writes one CSV per month into `outputs/`, re-reading any existing file first so already-saved URLs are skipped (resume support), with random 1.5-3.5 s delays and up to 3 retries per page.

In [ ]:
"""
Malay Mail Archive Scraper
Scrapes Malaysia news articles from 2015 to 2025 (all 12 months)
Saves: article_img, keyword, date, title, URL
"""

import requests
from bs4 import BeautifulSoup
import csv
import time
import random
import os
import sys
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
BASE_URL   = "https://www.malaymail.com/news/malaysia/{year}/{month:02d}"
OUTPUT_DIR = os.path.join(os.getcwd(), "outputs")

YEARS  = range(2015, 2026)   # 2015 → 2025 inclusive
MONTHS = range(1, 13)         # 01 → 12

KEYWORD = "malaysia"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

DELAY_MIN  = 1.5   # seconds between requests (be polite)
DELAY_MAX  = 3.5
MAX_RETRIES = 3

# ── Helpers ───────────────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update(HEADERS)


def get_page(url: str, retries: int = MAX_RETRIES) -> BeautifulSoup | None:
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=20)
            if resp.status_code == 200:
                return BeautifulSoup(resp.text, "html.parser")
            elif resp.status_code == 404:
                print(f"  [404] {url} – skipping")
                return None
            else:
                print(f"  [HTTP {resp.status_code}] attempt {attempt}/{retries}: {url}")
        except requests.RequestException as exc:
            print(f"  [ERROR] attempt {attempt}/{retries}: {exc}")
        if attempt < retries:
            time.sleep(DELAY_MAX * attempt)
    return None


def get_total_pages(soup: BeautifulSoup) -> int:
    """Read the last page number from the paginator."""
    pager = soup.select("ul.pager li.pager-nav a")
    max_page = 1
    for a in pager:
        href = a.get("href", "")
        text = a.get_text(strip=True)
        if text.isdigit():
            max_page = max(max_page, int(text))
    return max_page


def parse_articles(soup: BeautifulSoup) -> list[dict]:
    """Extract article cards from the archive page."""
    articles = []
    cards = soup.select("div.col-md-3.article-item")
    for card in cards:
        # ── Title & URL ──────────────────────────────────────────────────
        title_tag = card.select_one("h2.article-title a")
        if not title_tag:
            continue
        title = title_tag.get_text(strip=True)
        url   = title_tag.get("href", "").strip()
        if not url.startswith("http"):
            url = "https://www.malaymail.com" + url

        # ── Date ─────────────────────────────────────────────────────────
        date_tag = card.select_one("span.article-date")
        date_str = date_tag.get_text(strip=True) if date_tag else ""

        # ── Image ────────────────────────────────────────────────────────
        # Older articles use <img src="…"> directly; newer may use data-src
        img_tag = card.select_one("div.article-image img, div.layout-ratio img")
        img_url = ""
        if img_tag:
            img_url = (
                img_tag.get("src") or
                img_tag.get("data-src") or
                ""
            ).strip()
            # Skip the placeholder "no-image.png"
            if "no-image.png" in img_url:
                img_url = ""

        articles.append({
            "keyword":     KEYWORD,
            "date":        date_str,
            "title":       title,
            "url":         url,
            "article_img": img_url,
        })
    return articles


def scrape_month(year: int, month: int, writer: csv.DictWriter,
                 seen_urls: set) -> int:
    """Scrape all pages for a given year/month. Returns count of new rows."""
    base = BASE_URL.format(year=year, month=month)
    print(f"\n{'─'*60}")
    print(f"  Scraping {year}-{month:02d}  →  {base}")

    # Page 1 to discover total pages
    soup = get_page(f"{base}?pgno=1")
    if soup is None:
        print("  Could not load page 1 – skipping month")
        return 0

    total = get_total_pages(soup)
    print(f"  Total pages: {total}")

    count = 0
    soups = {1: soup}  # cache page 1

    for pgno in range(1, total + 1):
        if pgno in soups:
            pg_soup = soups[pgno]
        else:
            url = f"{base}?pgno={pgno}"
            pg_soup = get_page(url)
            if pg_soup is None:
                print(f"    Page {pgno}/{total}: failed – skipping")
                continue
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        articles = parse_articles(pg_soup)
        new = 0
        for art in articles:
            if art["url"] not in seen_urls:
                seen_urls.add(art["url"])
                writer.writerow(art)
                new += 1
        count += new
        print(f"    Page {pgno:3d}/{total}: +{new:3d} articles  (total new this month: {count})")

    return count


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    fieldnames = ["keyword", "date", "title", "url", "article_img"]

    grand_total = 0
    start_time  = datetime.now()

    for year in YEARS:
        for month in MONTHS:
            # ── One CSV per month ──────────────────────────────────────────
            output_csv = os.path.join(OUTPUT_DIR, f"malaymail_{year}_{month:02d}.csv")

            # Resume support: collect URLs already in this month's CSV
            seen_urls: set[str] = set()
            file_exists = os.path.isfile(output_csv)
            if file_exists:
                with open(output_csv, newline="", encoding="utf-8") as f:
                    reader = csv.DictReader(f)
                    for row in reader:
                        seen_urls.add(row.get("url", ""))
                print(f"[Resume] {output_csv}: {len(seen_urls)} existing URLs")

            try:
                with open(output_csv, "a", newline="", encoding="utf-8") as csvfile:
                    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                    if not file_exists:
                        writer.writeheader()

                    n = scrape_month(year, month, writer, seen_urls)
                    grand_total += n
            except KeyboardInterrupt:
                print("\n\n[Interrupted by user] Partial data saved.")
                sys.exit(0)
            except Exception as exc:
                print(f"  [UNEXPECTED ERROR] {year}-{month:02d}: {exc}")

    elapsed = datetime.now() - start_time
    print(f"\n{'='*60}")
    print(f"  Scraping complete.")
    print(f"  Total new articles saved : {grand_total}")
    print(f"  CSVs saved in            : {OUTPUT_DIR}")
    print(f"  Elapsed time             : {elapsed}")


if __name__ == "__main__":
    main()

[Output cleared — was a large execution log, removed to keep file size small]


In [16]:
pip install cloudscraper

Note: you may need to restart the kernel to use updated packages.


## Part 2

**Same URL collection, but through `cloudscraper`** to get past the 403 / Cloudflare block that stops Part 1. Identical card parsing, and it additionally writes a full request log (`scraper_log.csv`): start/end time, elapsed seconds, wait before request, year/month/page, URL, HTTP status and message, response type (HTML/JSON), detected site type (static vs. JS-rendered), response size in bytes, retry count and articles found per page.

In [18]:
"""
Malay Mail Archive Scraper — 2015 to 2025
1 CSV per year  →  output_malay/malaymail_YYYY.csv
Columns: keyword | date | title | url | article_img
Uses cloudscraper to bypass 403 blocks
"""

import cloudscraper
from bs4 import BeautifulSoup
import csv, time, random, os, sys
from datetime import datetime

# ── Config ──────────────────────────────────────────────────────────────────
BASE_URL   = "https://www.malaymail.com/news/malaysia/{year}/{month:02d}"
OUTPUT_DIR = "output_malay"
LOG_FILE   = os.path.join(OUTPUT_DIR, "scraper_log.csv")

YEARS   = range(2015, 2026)   # 2015 → 2025
MONTHS  = range(1, 13)
KEYWORD = "malaysia"

DELAY_MIN   = 2.0
DELAY_MAX   = 4.0
MAX_RETRIES = 3

scraper = cloudscraper.create_scraper()

# ── Log helpers ─────────────────────────────────────────────────────────────
log_fields = [
    "request_start", "request_end", "elapsed_sec",
    "wait_before_request_sec",                        # delay BETWEEN requests
    "year", "month", "page",
    "url", "http_status", "http_message",
    "response_type", "site_type",
    "content_bytes", "retries", "articles_found", "notes"
]

def write_log(row: dict):
    exists = os.path.isfile(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=log_fields)
        if not exists:
            w.writeheader()
        w.writerow(row)

def detect_site_type(html: str) -> str:
    for m in ["__NEXT_DATA__", "ng-app", "data-reactroot", "window.__INITIAL_STATE__"]:
        if m in html:
            return f"dynamic ({m})"
    if "article-title" in html or "article-item" in html:
        return "static"
    return "dynamic (no articles in raw HTML)"

http_msgs = {
    200: "OK", 301: "Moved Permanently", 403: "Forbidden",
    404: "Not Found", 429: "Too Many Requests", 500: "Server Error"
}

# ── Fetch ────────────────────────────────────────────────────────────────────
def get_page(url, year, month, pgno, wait_before=0.0):
    for attempt in range(1, MAX_RETRIES + 1):
        req_start = datetime.now()
        t0 = time.time()
        try:
            resp    = scraper.get(url, timeout=20)
            elapsed = round(time.time() - t0, 2)
            req_end = datetime.now()
            ct      = resp.headers.get("Content-Type", "")
            status  = resp.status_code
            row = {
                "request_start":          req_start.strftime("%Y-%m-%d %H:%M:%S"),
                "request_end":            req_end.strftime("%Y-%m-%d %H:%M:%S"),
                "elapsed_sec":            elapsed,
                "wait_before_request_sec": round(wait_before, 2),
                "year":           year,
                "month":          f"{month:02d}",
                "page":           pgno,
                "url":            url,
                "http_status":    status,
                "http_message":   http_msgs.get(status, "Unknown"),
                "response_type":  "JSON" if "json" in ct else "HTML",
                "site_type":      detect_site_type(resp.text),
                "content_bytes":  len(resp.content),
                "retries":        attempt - 1,
                "articles_found": 0,   # filled later
                "notes":          f"attempt {attempt}/{MAX_RETRIES}"
            }
            print(f"    pg{pgno:>3}  {status} {http_msgs.get(status,'?')}  "
                  f"elapsed:{elapsed}s  wait:{round(wait_before,2)}s  {len(resp.content)} bytes")

            if status == 200:
                soup = BeautifulSoup(resp.text, "html.parser")
                arts = parse_articles(soup)
                row["articles_found"] = len(arts)
                write_log(row)
                return soup
            elif status == 404:
                write_log(row)
                return None
            else:
                write_log(row)

        except Exception as e:
            elapsed = round(time.time() - t0, 2)
            req_end = datetime.now()
            print(f"    [Error] attempt {attempt}: {e}")
            write_log({
                "request_start":           req_start.strftime("%Y-%m-%d %H:%M:%S"),
                "request_end":             req_end.strftime("%Y-%m-%d %H:%M:%S"),
                "elapsed_sec":             elapsed,
                "wait_before_request_sec": round(wait_before, 2),
                "year": year, "month": f"{month:02d}", "page": pgno,
                "url": url, "http_status": "ERR", "http_message": str(e),
                "response_type": "-", "site_type": "-",
                "content_bytes": 0,
                "retries": attempt, "articles_found": 0,
                "notes": f"exception attempt {attempt}"
            })

        if attempt < MAX_RETRIES:
            time.sleep(DELAY_MAX * attempt)

    return None

# ── Parse ────────────────────────────────────────────────────────────────────
def parse_articles(soup) -> list:
    articles = []
    for card in soup.select("div.col-md-3.article-item"):
        a = card.select_one("h2.article-title a")
        if not a:
            continue
        title = a.get_text(strip=True)
        url   = a.get("href", "").strip()
        if not url.startswith("http"):
            url = "https://www.malaymail.com" + url
        date_tag = card.select_one("span.article-date")
        date     = date_tag.get_text(strip=True) if date_tag else ""
        img_tag  = card.select_one("div.article-image img, div.layout-ratio img")
        img = ""
        if img_tag:
            img = img_tag.get("src") or img_tag.get("data-src") or ""
            if "no-image" in img:
                img = ""
        articles.append({
            "keyword": KEYWORD, "date": date,
            "title": title, "url": url, "article_img": img
        })
    return articles

def get_total_pages(soup) -> int:
    max_page = 1
    for a in soup.select("ul.pager li.pager-nav a"):
        t = a.get_text(strip=True)
        if t.isdigit():
            max_page = max(max_page, int(t))
    return max_page

# ── Scrape one month ─────────────────────────────────────────────────────────
def scrape_month(year, month, writer, seen_urls) -> int:
    base  = BASE_URL.format(year=year, month=month)
    print(f"\n  ── {year}-{month:02d}  {base}")

    soup = get_page(f"{base}?pgno=1", year, month, 1, wait_before=0.0)
    if soup is None:
        print("  skipping month (page 1 failed)")
        return 0

    total = get_total_pages(soup)
    print(f"  total pages: {total}")

    count    = 0
    pg_cache = {1: soup}

    for pgno in range(1, total + 1):
        if pgno in pg_cache:
            pg_soup = pg_cache[pgno]
            wait    = 0.0
        else:
            wait    = random.uniform(DELAY_MIN, DELAY_MAX)
            time.sleep(wait)
            pg_soup = get_page(f"{base}?pgno={pgno}", year, month, pgno, wait_before=wait)
        if pg_soup is None:
            continue

        for art in parse_articles(pg_soup):
            if art["url"] not in seen_urls:
                seen_urls.add(art["url"])
                writer.writerow(art)
                count += 1

    print(f"  → {count} new articles")
    return count

# ── Main ─────────────────────────────────────────────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    fields     = ["keyword", "date", "title", "url", "article_img"]
    grand      = 0
    start_time = datetime.now()

    for year in YEARS:
        csv_path   = os.path.join(OUTPUT_DIR, f"malaymail_{year}.csv")
        seen_urls  = set()
        year_total = 0

        # resume: load already-saved URLs
        if os.path.isfile(csv_path):
            with open(csv_path, newline="", encoding="utf-8") as f:
                for row in csv.DictReader(f):
                    seen_urls.add(row.get("url", ""))
            print(f"[Resume] {csv_path}: {len(seen_urls)} existing URLs")

        print(f"\n{'='*60}")
        print(f"  YEAR {year}")
        print(f"{'='*60}")

        try:
            with open(csv_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fields, delimiter="\t")
                if not os.path.isfile(csv_path) or os.path.getsize(csv_path) == 0:
                    writer.writeheader()

                for month in MONTHS:
                    n = scrape_month(year, month, writer, seen_urls)
                    year_total += n
                    grand      += n

        except KeyboardInterrupt:
            print("\n[Interrupted] Partial data saved.")
            sys.exit(0)
        except Exception as e:
            print(f"  [ERROR] year {year}: {e}")

        print(f"\n  Year {year} total: {year_total} articles")

    elapsed = datetime.now() - start_time
    print(f"\n{'='*60}")
    print(f"  Done.  Total articles : {grand}")
    print(f"  Output folder        : {OUTPUT_DIR}/")
    print(f"  Log file             : {LOG_FILE}")
    print(f"  Scrape started       : {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Scrape ended         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  Elapsed              : {elapsed}")

main()

[Output cleared — was a large execution log, removed to keep file size small]


## Part 3

**Article-level scraper (first pass).** Reads the per-month URL lists produced above (from `INPUT_DIR`, starting at `malaymail_2015_07.csv`), opens every article URL, and extracts the full record: title (`h1.article-title`), published date and time (`article:published_time`), section, author, meta description, meta keywords, full body text (all `<p>` inside `div.article-body`, boilerplate lines dropped, order-preserving dedup) and `og:image`. Saves one `*_scraped.csv` per month into `scraped/` with the request-log fields as extra columns, retrying each URL up to 3 times and dumping a checkpoint CSV every 50 rows into `checkpoints/`.

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import random
import os
import glob
import json
from datetime import datetime

# ═══════════════════════════════════════════════════════════════════════
#  MALAY MAIL SCRAPER  (2015–2025, all months)
#  Mirrors Bernama v3: all log fields saved as columns in the same CSV
# ═══════════════════════════════════════════════════════════════════════

# ───────────────── INPUT / OUTPUT ─────────────────

# Folder that contains the per-month CSVs:
#   malaymail_2015_01.csv … malaymail_2025_12.csv
# Folder that contains the per-month CSVs:
#   malaymail_2015_01.csv … malaymail_2025_12.csv
INPUT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\outputs"

# Filter to start from 2015_07
START_FROM = "malaymail_2015_07.csv"
# Folder for per-month output CSVs  (one output file per input file)
OUTPUT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\scraped"

# Folder for checkpoint CSVs (every 50 articles inside one month's run)
CHECKPOINT_DIR = r"C:\Users\imt-medien\Desktop\Malay_Mail_2015_to_2025-20260604T153735Z-3-001\Malay_Mail_2015_to_2025\checkpoints"

# ───────────────── RETRY / CHECKPOINT CONFIG ─────────────────

MAX_RETRIES      = 3    # retries per URL
RETRY_DELAY      = 5    # seconds between retries
CHECKPOINT_EVERY = 50   # save checkpoint every N rows

# ───────────────── TEST MODE ─────────────────
# Set to True  → scrape only the 1st URL of the 1st CSV (quick test)
# Set to False → scrape ALL URLs across ALL CSVs
TEST_MODE = False

# ───────────────── REQUEST HEADERS ─────────────────

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/"
}

# ───────────────── STATUS CODE MESSAGES ─────────────────

STATUS_MESSAGES = {
    200: "OK",
    301: "Moved Permanently",
    302: "Found (Redirect)",
    403: "Forbidden",
    404: "Not Found",
    429: "Too Many Requests",
    500: "Internal Server Error",
    503: "Service Unavailable",
}

def get_status_msg(code):
    """Return a readable label for an HTTP status code."""
    return STATUS_MESSAGES.get(code, "Unknown Status")

# ───────────────── MALAY MAIL PARSER ─────────────────
#
# Page structure observed from the HTML:
#
#   TITLE        → <h1 class="article-title">
#   DATE/TIME    → <meta property="article:published_time" content="2015-01-31 22:44:57">
#                  also visible in <div class="article-date">Saturday, 31 Jan 2015 10:44 PM MYT</div>
#   SECTION      → <div class="article-section"><a href="/news/malaysia">Malaysia</a>
#   AUTHOR       → <meta name="author" content="...">  (often blank)
#                  also <div class="article-byline"> when present
#   ARTICLE TEXT → <div class="article-body"> → all <p> tags
#   IMAGE        → <meta property="og:image" content="...">
#                  (inline images inside article-body are ads/widgets; og:image is the clean one)
#   KEYWORDS     → <meta name="keywords" content="...">
#   DESCRIPTION  → <meta name="description" content="...">

def parse_article(html, url):
    """
    Parse a Malay Mail article page and return a dict of fields.
    Returns empty strings for any field not found.
    """
    soup = BeautifulSoup(html, "lxml")

    # ── TITLE ──
    # Primary: <h1 class="article-title">
    title = ""
    h1 = soup.find("h1", class_="article-title")
    if h1:
        title = h1.get_text(" ", strip=True)
    elif soup.find("h1"):
        title = soup.find("h1").get_text(" ", strip=True)
    elif soup.title:
        # last-resort: browser tab title, strip site name
        title = soup.title.get_text(" ", strip=True).replace("| Malay Mail", "").strip()

    # ── PUBLISHED DATE + TIME ──
    # <meta property="article:published_time" content="2015-01-31 22:44:57">
    published_date = ""
    published_time = ""
    meta_date = soup.find("meta", {"property": "article:published_time"})
    if meta_date:
        raw = meta_date.get("content", "").strip()   # "2015-01-31 22:44:57"
        parts = raw.split(" ")
        if len(parts) >= 1:
            published_date = parts[0]              # "2015-01-31"
        if len(parts) >= 2:
            published_time = " ".join(parts[1:])   # "22:44:57"

    # ── SECTION (category) ──
    # <div class="article-section"><a href="/news/malaysia">Malaysia</a>
    section = ""
    sec_div = soup.find("div", class_="article-section")
    if sec_div:
        sec_a = sec_div.find("a")
        if sec_a:
            section = sec_a.get_text(" ", strip=True)

    # ── AUTHOR ──
    # <meta name="author" content="Journalist Name"> or article-byline div
    author = ""
    meta_author = soup.find("meta", {"name": "author"})
    if meta_author:
        author = meta_author.get("content", "").strip()
    if not author:
        byline = soup.find("div", class_="article-byline")
        if byline:
            author = byline.get_text(" ", strip=True)

    # ── DESCRIPTION (meta) ──
    description = ""
    meta_desc = soup.find("meta", {"name": "description"})
    if meta_desc:
        description = meta_desc.get("content", "").strip()

    # ── KEYWORDS (meta) ──
    keywords = ""
    meta_kw = soup.find("meta", {"name": "keywords"})
    if meta_kw:
        keywords = meta_kw.get("content", "").strip()

    # ── OG IMAGE ──
    # Most reliable: og:image meta tag
    article_img = ""
    meta_img = soup.find("meta", {"property": "og:image"})
    if meta_img:
        article_img = meta_img.get("content", "").strip()

    # ── ARTICLE BODY TEXT ──
    # <div class="article-body"> contains the article <p> tags
    # Skip short/boilerplate paragraphs
    paragraphs = []
    article_body = soup.find("div", class_="article-body")

    if article_body:
        # Use only <p> tags inside article-body.
        # NO minimum-length filter — short closing lines like "— Bernama" must be kept.
        for p in article_body.find_all("p"):
            text = p.get_text(" ", strip=True)
            if not text:
                continue
            # skip only true navigation/boilerplate lines
            # "— Bernama" is intentionally NOT in this list so it is always kept
            bad_patterns = [
                "Follow us on",
                "Subscribe to",
                "You May Also Like",
                "Related Articles",
            ]
            if any(x in text for x in bad_patterns):
                continue
            paragraphs.append(text)
    else:
        # Fallback: grab all <p> from the full page (less clean)
        for p in soup.find_all("p"):
            text = p.get_text(" ", strip=True)
            if text:
                paragraphs.append(text)

    # deduplicate while preserving order
    paragraphs = list(dict.fromkeys(paragraphs))
    article_text = "\n\n".join(paragraphs)

    return {
        "keyword":        "Malaysia",     # fixed keyword for all Malay Mail rows
        "published_date": published_date,
        "published_time": published_time,
        "section":        section,
        "author":         author,
        "title":          title,
        "description":    description,
        "keywords":       keywords,
        "article_text":   article_text,
        "article_img":    article_img,
        "url":            url,
    }

# ───────────────── CHECKPOINT SAVE ─────────────────

def save_checkpoint(rows, checkpoint_path):
    """Append rows to (or create) a checkpoint CSV."""
    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)
    pd.DataFrame(rows).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
    print(f"  [CHECKPOINT] {len(rows)} rows → {checkpoint_path}")

# ───────────────── COLLECT INPUT CSVs ─────────────────

# Pattern: malaymail_YYYY_MM.csv  (2015_01 … 2025_12)
csv_pattern = os.path.join(INPUT_DIR, "malaymail_*.csv")
all_csv_files = sorted(glob.glob(csv_pattern))

# Filter to start from 2015_07
all_csv_files = [f for f in all_csv_files if os.path.basename(f) >= START_FROM]

if not all_csv_files:
    print(f"ERROR: No CSV files found matching: {csv_pattern}")
    exit(1)

print(f"\nFound {len(all_csv_files)} input CSV files")
os.makedirs(OUTPUT_DIR,     exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ───────────────── SCRIPT START TIME ─────────────────

script_start     = time.time()
script_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"SCRIPT STARTED: {script_start_str}")

grand_total_rows = 0   # across all CSV files

# ═══════════════════════════════════════════════════════════════════════
#  OUTER LOOP — one CSV file at a time
# ═══════════════════════════════════════════════════════════════════════

for csv_idx, csv_path in enumerate(all_csv_files, start=1):

    csv_name = os.path.basename(csv_path)   # e.g. "malaymail_2015_01.csv"
    csv_stem = os.path.splitext(csv_name)[0]

    print("\n" + "█" * 70)
    print(f"[CSV {csv_idx}/{len(all_csv_files)}] {csv_name}")

    # ── read URLs from this month's CSV ──
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"  ERROR reading CSV: {e} — skipping")
        continue

    if "url" not in df.columns:
        print(f"  WARNING: no 'url' column in {csv_name} — skipping")
        continue

    urls = df["url"].dropna().unique().tolist()

    # TEST MODE: only 1st URL of the 1st CSV file
    if TEST_MODE:
        if csv_idx == 1:
            urls = urls[:1]
            print(f"  TEST MODE: using 1 URL")
        else:
            print(f"  TEST MODE: skipping remaining CSVs")
            break

    print(f"  URLs to scrape: {len(urls)}")

    # ── output paths for this month ──
    output_csv        = os.path.join(OUTPUT_DIR,     f"{csv_stem}_scraped.csv")
    checkpoint_prefix = os.path.join(CHECKPOINT_DIR, f"{csv_stem}_ckpt")

    all_rows         = []   # all rows for this month
    checkpoint_buf   = []   # buffer since last checkpoint
    checkpoint_idx   = 1    # checkpoint counter for this month
    total_urls       = len(urls)

    # ═══════════════════════════════════════════════════════════════════
    #  INNER LOOP — one URL at a time
    # ═══════════════════════════════════════════════════════════════════

    for url_idx, url in enumerate(urls, start=1):

        print("\n" + "=" * 70)
        print(f"  [{url_idx}/{total_urls}] FETCHING")
        print(f"  URL: {url}")

        # ── per-request tracking variables (reset each URL) ──
        response          = None
        attempt           = 0
        request_start_str = ""
        request_end_str   = ""
        request_duration  = 0.0
        status_code       = ""
        status_msg        = ""
        redirected        = "No"
        redirect_hops     = ""
        final_url         = url
        req_headers_str   = ""
        resp_headers_str  = ""
        wait_time         = 0.0
        total_attempts    = 0
        error_msg         = ""
        scrape_status     = "SKIPPED"

        while attempt < MAX_RETRIES:

            attempt += 1
            print(f"    ATTEMPT: {attempt}/{MAX_RETRIES}")

            # record request start time
            request_start     = time.time()
            request_start_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"    REQUEST START: {request_start_str}")

            try:

                response = requests.get(
                    url,
                    headers=HEADERS,
                    timeout=30,
                    allow_redirects=True    # follow redirects automatically
                )

                # record end time
                request_end      = time.time()
                request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                request_duration = round(request_end - request_start, 2)

                print(f"    REQUEST END:   {request_end_str}")
                print(f"    REQUEST TIME:  {request_duration}s")

                # ── status code + message ──
                status_code = response.status_code
                status_msg  = get_status_msg(status_code)
                print(f"    STATUS: {status_code} — {status_msg}")

                # ── redirect detection ──
                if response.history:
                    redirected    = f"Yes ({len(response.history)} hop(s))"
                    redirect_hops = " | ".join(
                        f"{h.status_code} → {h.url}" for h in response.history
                    )
                    final_url = response.url
                    print(f"    REDIRECTED: {redirected}")
                    for i, hop in enumerate(response.history, 1):
                        print(f"      hop {i}: {hop.status_code} → {hop.url}")
                    print(f"    FINAL URL: {final_url}")
                else:
                    print(f"    REDIRECTED: No")

                # ── capture headers as JSON strings for CSV ──
                req_headers_str  = json.dumps(dict(response.request.headers), ensure_ascii=False)
                resp_headers_str = json.dumps(dict(response.headers),         ensure_ascii=False)

                # ── print request headers ──
                print("    REQUEST HEADERS:")
                for k, v in response.request.headers.items():
                    print(f"      {k}: {v}")

                # ── print response headers ──
                print("    RESPONSE HEADERS:")
                for k, v in response.headers.items():
                    print(f"      {k}: {v}")

                # success → exit retry loop
                if status_code == 200:
                    break

                # retryable server-side errors
                if status_code in (429, 500, 503):
                    print(f"    Retryable {status_code}. Waiting {RETRY_DELAY}s...")
                    time.sleep(RETRY_DELAY)
                else:
                    # 404, 403, etc. → no point retrying
                    print(f"    Non-retryable {status_code}. Skipping.")
                    break

            except Exception as e:
                request_end      = time.time()
                request_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                request_duration = round(request_end - request_start, 2)
                error_msg        = str(e)
                scrape_status    = "ERROR"
                print(f"    ERROR on attempt {attempt}: {error_msg}")
                print(f"    REQUEST TIME (before error): {request_duration}s")
                if attempt < MAX_RETRIES:
                    print(f"    Waiting {RETRY_DELAY}s before retry...")
                    time.sleep(RETRY_DELAY)

        total_attempts = attempt   # how many tries actually happened

        # ── parse if we got a 200 ──
        if response is not None and response.status_code == 200:

            html  = response.text
            soup  = BeautifulSoup(html, "lxml")

            print(f"    TITLE CHECK: {soup.title}")
            print(f"    P TAGS: {len(soup.find_all('p'))}")

            article = parse_article(html, url)

            print(f"    TITLE:        {article['title'][:100]}")
            print(f"    SECTION:      {article['section']}")
            print(f"    AUTHOR:       {article['author']}")
            print(f"    DATE:         {article['published_date']}")
            print(f"    TIME:         {article['published_time']}")
            print(f"    IMAGE FOUND:  {'Yes' if article['article_img'] else 'No'}")
            print(f"    TEXT CHARS:   {len(article['article_text'])}")

            scrape_status = "SUCCESS"

        else:
            # blank article fields so the row still exists in the CSV
            article = {
                "keyword":        "Malaysia",
                "published_date": "",
                "published_time": "",
                "section":        "",
                "author":         "",
                "title":          "",
                "description":    "",
                "keywords":       "",
                "article_text":   "",
                "article_img":    "",
                "url":            url,
            }
            print(f"    SKIPPED (no valid response after {total_attempts} attempt(s))")

        # ── wait between requests ──
        wait_time = round(random.uniform(5, 10), 1)
        print(f"\n    WAITING {wait_time}s before next request...")
        time.sleep(wait_time)

        # ── merge article + all log fields into one CSV row ──
        row = {
            # ── article content fields ──
            **article,

            # ── request timing ──
            "log_request_start":      request_start_str,
            "log_request_end":        request_end_str,
            "log_request_duration_s": request_duration,

            # ── retry info ──
            "log_total_attempts":     total_attempts,
            "log_max_retries":        MAX_RETRIES,

            # ── wait time after this request ──
            "log_wait_time_s":        wait_time,

            # ── HTTP status ──
            "log_status_code":        status_code,
            "log_status_msg":         status_msg,

            # ── redirect info ──
            "log_redirected":         redirected,
            "log_redirect_hops":      redirect_hops,
            "log_final_url":          final_url,

            # ── headers (JSON strings) ──
            "log_request_headers":    req_headers_str,
            "log_response_headers":   resp_headers_str,

            # ── error + result ──
            "log_error_msg":          error_msg,
            "log_scrape_status":      scrape_status,
        }

        all_rows.append(row)
        checkpoint_buf.append(row)

        # ── checkpoint every N rows ──
        if len(checkpoint_buf) >= CHECKPOINT_EVERY:
            ckpt_path = f"{checkpoint_prefix}_{checkpoint_idx:04d}.csv"
            save_checkpoint(checkpoint_buf, ckpt_path)
            checkpoint_idx += 1
            checkpoint_buf  = []

    # ── save leftover rows from this month ──
    if checkpoint_buf:
        ckpt_path = f"{checkpoint_prefix}_{checkpoint_idx:04d}.csv"
        save_checkpoint(checkpoint_buf, ckpt_path)

    # ── save final output CSV for this month ──
    if all_rows:
        month_df = pd.DataFrame(all_rows)
        month_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
        success_count = (month_df["log_scrape_status"] == "SUCCESS").sum()
        print(f"\n  [SAVED] {csv_name} → {output_csv}")
        print(f"  Rows: {len(month_df)}  |  SUCCESS: {success_count}  |  FAILED: {len(month_df) - success_count}")
        grand_total_rows += len(month_df)
    else:
        print(f"  No rows collected for {csv_name}")

# ───────────────── SCRIPT END TIME ─────────────────

script_end      = time.time()
script_end_str  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
total_duration  = round(script_end - script_start, 2)

print("\n" + "═" * 70)
print("ALL DONE")
print(f"SCRIPT STARTED:    {script_start_str}")
print(f"SCRIPT ENDED:      {script_end_str}")
print(f"TOTAL TIME:        {total_duration}s  ({round(total_duration/60, 2)} min)")
print(f"TOTAL ROWS SAVED:  {grand_total_rows}")
print(f"OUTPUT DIR:        {OUTPUT_DIR}")
print("═" * 70)

[Output cleared — was a large execution log, removed to keep file size small]
